# Visualizing VE, VP, and linear diffusion paths

This notebook compares three 2D trajectories with one frozen data sample $x_0$ and one frozen noise sample $\epsilon$.

VE:
$$
x_t = x_0 + t\epsilon
$$

VP:
$$
x_t = \alpha_t x_0 + \sigma_t \epsilon
$$

Linear interpolation:
$$
x_t = (1-t)x_0 + t\epsilon
$$

Use the sliders below to move through time and change $x_0$, $\epsilon$, and the VE noise scale.

In [ ]:
## WSL notes

From this folder, JupyterLab can be launched with:

```bash
jupyter lab --no-browser --ip 127.0.0.1
```

Open the localhost URL printed by Jupyter in Windows. If imports fail in a fresh environment, install the notebook dependencies with:

```bash
python3 -m pip install numpy matplotlib ipywidgets jupyterlab ipykernel
```

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print("Ready: numpy, matplotlib, and ipywidgets imported successfully.")

In [ ]:
def vp_alpha_sigma(t, schedule="sqrt"):
    """Return alpha_t and sigma_t with alpha_t**2 + sigma_t**2 = 1."""
    t = np.asarray(t, dtype=float)

    if schedule == "sqrt":
        alpha = np.sqrt(np.clip(1.0 - t, 0.0, 1.0))
        sigma = np.sqrt(np.clip(t, 0.0, 1.0))
    elif schedule == "cosine":
        alpha = np.cos(0.5 * np.pi * t)
        sigma = np.sin(0.5 * np.pi * t)
    else:
        raise ValueError(f"Unknown VP schedule: {schedule}")

    return alpha, sigma


def ve_xt(x0, eps, t, noise_scale=1.0):
    t = np.asarray(t, dtype=float)
    return x0 + (noise_scale * t)[..., None] * eps


def vp_xt(x0, eps, t, schedule="sqrt"):
    alpha, sigma = vp_alpha_sigma(t, schedule=schedule)
    return alpha[..., None] * x0 + sigma[..., None] * eps


def linear_xt(x0, eps, t):
    t = np.asarray(t, dtype=float)
    return (1.0 - t)[..., None] * x0 + t[..., None] * eps


def build_paths(x0, eps, ts, noise_scale=1.0, vp_schedule="sqrt"):
    return {
        "VE": ve_xt(x0, eps, ts, noise_scale=noise_scale),
        "VP": vp_xt(x0, eps, ts, schedule=vp_schedule),
        "Linear": linear_xt(x0, eps, ts),
    }

In [ ]:
COLORS = {
    "VE": "#1f77b4",
    "VP": "#d62728",
    "Linear": "#2ca02c",
}


def plot_diffusion_paths(
    t=0.35,
    x0_x=2.0,
    x0_y=1.0,
    eps_x=-1.0,
    eps_y=2.0,
    ve_noise_scale=1.0,
    vp_schedule="sqrt",
    steps=101,
    show_trace=True,
):
    x0 = np.array([x0_x, x0_y], dtype=float)
    eps = np.array([eps_x, eps_y], dtype=float)
    ts = np.linspace(0.0, 1.0, int(steps))
    paths = build_paths(x0, eps, ts, noise_scale=ve_noise_scale, vp_schedule=vp_schedule)
    current = build_paths(x0, eps, np.array(t), noise_scale=ve_noise_scale, vp_schedule=vp_schedule)

    fig, ax = plt.subplots(figsize=(6.8, 6.2))

    if show_trace:
        for name, path in paths.items():
            ax.plot(path[:, 0], path[:, 1], color=COLORS[name], linewidth=2.0, label=f"{name} path")
            ax.scatter(path[:: max(1, len(path) // 10), 0], path[:: max(1, len(path) // 10), 1], color=COLORS[name], s=12, alpha=0.6)

    ax.scatter([x0[0]], [x0[1]], color="black", s=70, marker="o", label="$x_0$")
    ax.scatter([eps[0]], [eps[1]], color="#9467bd", s=70, marker="s", label="$\\epsilon$")
    ax.scatter([0], [0], color="0.45", s=35, marker="+", label="origin")

    for name, point in current.items():
        point = np.asarray(point).reshape(2)
        ax.scatter([point[0]], [point[1]], color=COLORS[name], edgecolor="white", linewidth=1.2, s=130, zorder=5)
        ax.annotate(name, point, xytext=(7, 7), textcoords="offset points", color=COLORS[name], weight="bold")

    all_points = np.vstack([x0, eps, np.zeros(2), *paths.values()])
    xy_min = all_points.min(axis=0)
    xy_max = all_points.max(axis=0)
    span = max(float(xy_max[0] - xy_min[0]), float(xy_max[1] - xy_min[1]), 1.0)
    center = 0.5 * (xy_min + xy_max)
    pad = 0.25 * span
    ax.set_xlim(center[0] - 0.5 * span - pad, center[0] + 0.5 * span + pad)
    ax.set_ylim(center[1] - 0.5 * span - pad, center[1] + 0.5 * span + pad)
    ax.set_aspect("equal", adjustable="box")
    ax.axhline(0, color="0.8", linewidth=1)
    ax.axvline(0, color="0.8", linewidth=1)
    ax.set_xlabel("x dimension 1")
    ax.set_ylabel("x dimension 2")
    ax.set_title(f"2D diffusion trajectories at t = {t:.2f}")
    ax.legend(loc="best", fontsize=8)
    plt.show()

    for name, point in current.items():
        point = np.asarray(point).reshape(2)
        print(f"{name:>6} x_t = ({point[0]: .3f}, {point[1]: .3f})")

In [ ]:
slider_style = {"description_width": "110px"}
slider_layout = widgets.Layout(width="340px")

controls = {
    "t": widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.01, description="t", readout_format=".2f", continuous_update=False, layout=slider_layout, style=slider_style),
    "x0_x": widgets.FloatSlider(value=2.0, min=-4.0, max=4.0, step=0.1, description="x0 dim 1", continuous_update=False, layout=slider_layout, style=slider_style),
    "x0_y": widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description="x0 dim 2", continuous_update=False, layout=slider_layout, style=slider_style),
    "eps_x": widgets.FloatSlider(value=-1.0, min=-4.0, max=4.0, step=0.1, description="eps dim 1", continuous_update=False, layout=slider_layout, style=slider_style),
    "eps_y": widgets.FloatSlider(value=2.0, min=-4.0, max=4.0, step=0.1, description="eps dim 2", continuous_update=False, layout=slider_layout, style=slider_style),
    "ve_noise_scale": widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.1, description="VE scale", continuous_update=False, layout=slider_layout, style=slider_style),
    "vp_schedule": widgets.Dropdown(options=[("sqrt", "sqrt"), ("cosine", "cosine")], value="sqrt", description="VP schedule", layout=slider_layout, style=slider_style),
    "steps": widgets.IntSlider(value=101, min=11, max=401, step=10, description="trace steps", continuous_update=False, layout=slider_layout, style=slider_style),
    "show_trace": widgets.Checkbox(value=True, description="show full paths", indent=False, layout=widgets.Layout(width="180px")),
}

ui = widgets.VBox([
    widgets.HBox([widgets.VBox([controls["t"], controls["ve_noise_scale"], controls["vp_schedule"], controls["steps"], controls["show_trace"]]),
                  widgets.VBox([controls["x0_x"], controls["x0_y"], controls["eps_x"], controls["eps_y"]])])
])

out = widgets.interactive_output(plot_diffusion_paths, controls)
display(ui, out)

## What to look for

- VE starts at $x_0$ and moves in the noise direction. With this simple schedule, it does not end at $\epsilon$ unless the vectors happen to line up that way.
- VP rotates the balance from data to noise while keeping $\alpha_t^2 + \sigma_t^2 = 1$.
- Linear interpolation follows the straight segment from $x_0$ to $\epsilon$.